In [13]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from collections import deque
import time

START_URL = "http://a.rw"   # upgraded to https
MAX_PAGES = 200
DELAY = 1.0

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0 Safari/537.36"
    )
})

def same_domain(url, domain):
    try:
        return urlparse(url).netloc == domain
    except Exception:
        return False

def normalize_scheme(url):
    """Force https for this domain if it's http."""
    parsed = urlparse(url)
    if parsed.scheme == "http":
        parsed = parsed._replace(scheme="https")
    return parsed.geturl()

def crawl(start_url):
    parsed_start = urlparse(start_url)
    domain = parsed_start.netloc

    to_visit = deque([start_url])
    visited = set()

    while to_visit and len(visited) < MAX_PAGES:
        url = to_visit.popleft()
        if url in visited:
            continue

        print(f"[{len(visited)+1}] Fetching:", url)
        visited.add(url)

        try:
            resp = session.get(url, timeout=10)
        except requests.exceptions.ConnectionError as e:
            print("  ! Connection error:", e)
            continue
        except requests.RequestException as e:
            print("  ! Request failed:", e)
            continue

        if resp.status_code >= 400:
            print(f"  ! HTTP {resp.status_code}")
            continue

        html = resp.text
        save_page(url, html)

        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            href = a["href"]
            next_url = urljoin(url, href)
            next_url = normalize_scheme(next_url)

            parsed = urlparse(next_url)
            normalized = parsed._replace(fragment="").geturl()

            if parsed.scheme in ("http", "https") and same_domain(normalized, domain):
                if normalized not in visited:
                    to_visit.append(normalized)

        time.sleep(DELAY)

def save_page(url, html):
    from pathlib import Path
    parsed = urlparse(url)
    path = parsed.path
    if not path or path.endswith("/"):
        path = path + "index.html"

    filename = (parsed.netloc + path).replace("/", "_")
    Path("pages").mkdir(exist_ok=True)
    filepath = Path("pages") / filename

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(html)
    print("  -> Saved as", filepath)

if __name__ == "__main__":
    crawl(START_URL)


[1] Fetching: http://a.rw
  ! HTTP 403


In [15]:
import http.client
from urllib.parse import urlencode
from bs4 import BeautifulSoup

conn = http.client.HTTPSConnection("api.scrapingant.com")

params = {
    "url": "http://a.rw",
    "x-api-key": "361abfb9da0e49e898963d9b6ac45b1a"
}

conn.request("GET", f"/v2/general?{urlencode(params)}")

res = conn.getresponse()
data = res.read()

# Parse HTML and extract text
soup = BeautifulSoup(data.decode("utf-8"), "html.parser")

# Remove script and style elements
for script in soup(["script", "style"]):
    script.decompose()

# Get text
text = soup.get_text()

# Clean up whitespace
lines = (line.strip() for line in text.splitlines())
chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
text = '\n'.join(chunk for chunk in chunks if chunk)

print(text)

a.rw: The domain name a.rw is for sale
Squadhelp is now Atom -- where everything starts!
Learn More
My Dashboard
My Account
Logout
Login
Signup
(877) 355-3585
Chat
Email
Help Desk
Excellent
Trustpilot
Domains for Sale
Premium Domain Marketplace
Explore 300,000+ expert-curated, brandable domains to elevate your business.
Ultra Premium Marketplace
Discover the world’s most coveted and powerful domains for top-tier brands.
Sapphire Marketplace
Find one-word domains with modern extensions like .ai, .io, and .xyz.
Top Domain Collections
.ai Domains Popular
Short Domains
One-Word Domains
3 Letter Domains
4 Letter Domains
5 Letter Domains
Country-Specific Domains
Domain Services
Domain Transactions AtomPay
Domain Broker
Domain Auction
Get Started
Find your perfect domain today and buy instantly in the Atom.com marketplace.
Free Domain Tools
AI Domain Name Generator
Get hundreds of smart domain ideas in seconds.
AI Domain Appraisal Tool
Instantly check your domain’s market
value.
WHOIS Domain 

In [ ]:
import csv
import itertools
import re
import string
import http.client
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from urllib.parse import urlencode

from bs4 import BeautifulSoup

# Increase CSV field size limit to handle large text content
csv.field_size_limit(10485760)  # 10MB limit

# ----------------- CONFIG -----------------

TLD_FILE = "data/two_letter_tlds.txt"
OUTPUT_TEXT_CSV = "domain_text_scrape.csv"
SCRAPINGANT_API_KEY = "361abfb9da0e49e898963d9b6ac45b1a"

ignored_tlds = {".bl", ".bq", ".eh"}

MAX_WORKERS = 1   # Lower for API rate limits

# ------------------------------------------


def load_tlds(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return [
            line.strip().lower()
            for line in f
            if line.strip() and line.strip().lower() not in ignored_tlds
        ]


def generate_domains(tlds, length: int = 1):
    base_chars = list(string.ascii_lowercase) + list(string.digits)
    labels = base_chars if length == 1 else [
        "".join(p) for p in itertools.product(base_chars, repeat=length)
    ]

    for label in labels:
        for tld in tlds:
            yield f"{label}{tld}"


def fetch_page_scrapingant(domain: str):
    """Fetch page using ScrapingAnt API."""
    urls = [f"http://{domain}"]

    for url in urls:
        try:
            conn = http.client.HTTPSConnection("api.scrapingant.com")
            
            params = {
                "url": url,
                "x-api-key": SCRAPINGANT_API_KEY,
                "browser": False,
            }
            
            conn.request("GET", f"/v2/general?{urlencode(params)}")
            
            res = conn.getresponse()
            
            # DEBUG: Print status and headers
            print(f"\n=== DEBUG for {url} ===")
            print(f"Status: {res.status}")
            print(f"Headers: {dict(res.getheaders())}")
            
            data = res.read()
            
            # DEBUG: Print raw response (first 500 chars)
            print(f"Response preview: {data[:500]}")
            
            if res.status == 200:
                html = data.decode("utf-8")
                conn.close()
                return url, html
            else:
                # DEBUG: Print full error response
                print(f"Error response body: {data.decode('utf-8', errors='ignore')}")
                conn.close()
            
        except Exception as e:
            print(f"  Error fetching {url}: {e}")
            continue
    
    return None, None


def extract_visible_text(html: str) -> str:
    """Extract clean text from HTML."""
    soup = BeautifulSoup(html, "html.parser")
    
    # Remove script and style elements
    for tag in soup(["script", "style", "noscript", "iframe", "header", "footer"]):
        tag.decompose()
    
    # Get text
    text = soup.get_text()
    
    # Clean up whitespace
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    text = ' '.join(chunk for chunk in chunks if chunk)
    
    return text.strip()


def load_existing_csv(path: str):
    if not Path(path).is_file():
        return {}

    existing = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            domain = row["domain"].lower()
            existing[domain] = {"url": row["url"], "text": row["text"]}
    return existing


def save_csv(path: str, data: dict):
    """Rewrite entire CSV safely."""
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["domain", "url", "text"])
        for domain, info in sorted(data.items()):
            writer.writerow([domain, info["url"], info["text"]])


def scrape_domain(domain: str):
    """Threaded worker function using ScrapingAnt."""
    url, html = fetch_page_scrapingant(domain)
    if not url or not html:
        return domain, "", ""
    text = extract_visible_text(html)
    return domain, url, text


# ------------- MAIN (MULTITHREADED) -------------


if __name__ == "__main__":
    tlds = load_tlds(TLD_FILE)

    existing = load_existing_csv(OUTPUT_TEXT_CSV)
    print(f"Loaded {len(existing)} existing entries.")

    lock = Lock()

    all_domains = list(generate_domains(tlds, length=1))

    # Only scrape domains that need it
    domains_to_scrape = [
        d for d in all_domains
        if d not in existing or not existing[d]["text"]
    ]

    print(f"Need to scrape: {len(domains_to_scrape)} domains")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(scrape_domain, dom): dom
            for dom in domains_to_scrape
        }

        for i, future in enumerate(as_completed(futures), 1):
            domain, url, text = future.result()

            with lock:
                existing[domain] = {"url": url, "text": text}
                save_csv(OUTPUT_TEXT_CSV, existing)

            print(f"[{i}/{len(domains_to_scrape)}] Updated {domain} (text_length={len(text)})")

    print(f"\n✓ Complete! Results saved to {OUTPUT_TEXT_CSV}")

Loaded 21 existing entries.
Need to scrape: 9069 domains

=== DEBUG for http://a.ac ===
Status: 200
Headers: {'Ant-Credits-Cost': '1', 'Ant-Original-Header-Alt-Svc': 'h3=":443"; ma=86400', 'Ant-Original-Header-Connection': 'keep-alive', 'Ant-Original-Header-Content-Encoding': 'br', 'Ant-Original-Header-Content-Type': 'text/html', 'Ant-Original-Header-Date': 'Sun, 23 Nov 2025 03:45:17 GMT', 'Ant-Original-Header-Etag': 'W/"629bd75d-9d2"', 'Ant-Original-Header-Last-Modified': 'Sat, 04 Jun 2022 22:06:21 GMT', 'Ant-Original-Header-Referrer-Policy': 'strict-origin-when-cross-origin', 'Ant-Original-Header-Server': 'Microsoft-IIS/88.88', 'Ant-Original-Header-Strict-Transport-Security': 'max-age=63072000; includeSubDomains; preload', 'Ant-Original-Header-Transfer-Encoding': 'chunked', 'Ant-Original-Header-Vary': 'Accept-Encoding', 'Ant-Original-Header-X-Content-Type-Options': 'nosniff', 'Ant-Original-Header-X-Frame-Options': 'SAMEORIGIN', 'Ant-Original-Header-X-Protocol': 'HTTP/1.1', 'Ant-Origi